# Apt 305 — the complete paper figure set

Builds all ten figures (F1–F10) into `results/paper/figures/`, as **PDF** for the
paper and **300 dpi PNG** for preview, plus `FIGURES.md`.

## What this notebook does not do

**It does not run the engine.** Every number in every figure is read from a file
already committed under `results/` — `trajectory_v2/`, `baseline_vs_ep_v2/` and
`results/diagnostics/`. That is the point: the figures and the tables in the paper
are then the *same measurements*, not two runs that happen to agree. A cell that
re-simulated would break that guarantee, so there isn't one.

## The metric

Every figure that prints a per-area value uses

$$Q_{need} = Q_{H,sens} + Q_{C,sens} + Q_{C,lat}\ \text{(gated)}$$

— sensible heating + sensible cooling + gated latent cooling, excluding latent
heating and excluding the ungated moisture balance. This is the
`Total, sensible + gated latent` column of `trajectory_v2/comparison.md`, **not**
the engine's own total column; the two differ on the four states before the latent
correction by the ~153–185 kWh of phantom humidification.

`tools/figures/figstyle.py` recomputes all thirteen per-area values and the
canonical headline from the raw JSON and asserts them against the methodology's
published list **before any figure is drawn**. Section 2 below runs that gate on
its own, and then tampers with it to show it is live.

**Canonical state** `+Closure fixes` — 123.74 kWh sensible heating, 13.41 kWh
sensible cooling, 1.14 kWh gated latent, 138.29 kWh total, **6.91 kWh/m²·yr**.

**Weather** `AUS_VIC_Melbourne-Essendon.Fields.958660_TMYx.2011-2025.epw`


## 1 · Setup


In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO   = Path('/content/AIB')
BRANCH = 'claude/new-session-carh9p'   # the branch carrying tools/figures/

if not REPO.exists():
    subprocess.run(['git', 'clone',
                    'https://github.com/samiraghafarigousheh-sys/aib.git', str(REPO)],
                   check=True)
os.chdir(REPO)

subprocess.run(['git', 'fetch', 'origin', BRANCH], check=True)
subprocess.run(['git', 'checkout', BRANCH], check=True)
subprocess.run(['git', 'pull', '--ff-only', 'origin', BRANCH], check=False)

# The figure set needs matplotlib and numpy and nothing else -- no engine import,
# no plotly, no pyecharts. Colab ships both, so this is a no-op there and a real
# install anywhere else.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'matplotlib>=3.6', 'numpy'], check=True)

import matplotlib, numpy
print('repo    ', REPO)
print('branch  ', subprocess.run(['git','rev-parse','--abbrev-ref','HEAD'],
                                 capture_output=True, text=True).stdout.strip())
print('commit  ', subprocess.run(['git','rev-parse','--short','HEAD'],
                                 capture_output=True, text=True).stdout.strip())
print('matplotlib', matplotlib.__version__, '· numpy', numpy.__version__)


### The source files the figures read

Listed before anything is built, so a missing input is visible here rather than as
a traceback three cells later.


In [ ]:
SOURCES = [
    'results/paper/trajectory_v2/trajectory_raw.json',
    'results/paper/trajectory_v2/comparison.md',
    'results/paper/baseline_vs_ep_v2/baseline_vs_energyplus.csv',
    'results/paper/baseline_vs_ep_v2/run_meta.json',
    'results/diagnostics/wind_stats_essendon.json',
    'results/diagnostics/wind_stats.json',
    'results/au_corrections_closed/six_state_closed.md',
    'weather_cache/AUS_VIC_Melbourne-Essendon.Fields.958660_TMYx.2011-2025.epw',
]
missing = [s for s in SOURCES if not Path(s).exists()]
for s in SOURCES:
    p = Path(s)
    mark = 'ok  ' if p.exists() else 'MISS'
    size = f'{p.stat().st_size:>10,} B' if p.exists() else ' ' * 12
    print(f'{mark} {size}  {s}')
assert not missing, f'missing source files: {missing}'


## 2 · The gate, run on its own

`figstyle.load_trajectory()` reads the raw JSON, recomputes
`(Q_H_sens + Q_C_sens + Q_C_lat_gated) / 20 m²` for all thirteen states, and
compares each against the value the methodology publishes. It also checks the
canonical headline, the state order, and that the run really was on the Essendon
file. `load_ep()` does the same for the three ISO-vs-EnergyPlus differences,
recomputed against the **EnergyPlus** reference (the committed CSV states them
against ISO).

Nothing is plotted here. This is the check on its own, so you can see it pass.


In [ ]:
sys.path.insert(0, str(REPO / 'tools' / 'figures'))
import figstyle as F

traj = F.load_trajectory()
ep   = F.load_ep()

print(f"{'state':<30}{'H sens':>10}{'C sens':>9}{'lat gated':>11}"
      f"{'kWh/m2':>9}{'paper':>8}")
print('-' * 77)
for s in F.STATES:
    d = traj['states'][s]
    print(f"{s:<30}{d['Q_H_sens']:>10.2f}{d['Q_C_sens']:>9.2f}"
          f"{d['Q_C_lat_gated']:>11.2f}{d['Q_need_per_sqm']:>9.2f}"
          f"{F.EXPECTED_PER_AREA[s]:>8.2f}")
print('-' * 77)
print('all thirteen per-area values agree with the methodology list.')
print()
for m in ('Heating', 'Cooling', 'Total'):
    print(f"{m:<9} ISO {ep[m]['iso']:>9,.1f} kWh   EnergyPlus {ep[m]['ep']:>9,.1f} kWh"
          f"   {ep[m]['diff_pct_vs_ep']:+6.1f} % vs EP")


### The hard stop

A gate you never see fire is a gate you are trusting on faith. The cell below
corrupts one expected value in memory and calls the loader again. It must raise
`MissingQuantity` and name the state — if it returns quietly, the gate is not
protecting anything and nothing below should be believed.


In [ ]:
real = F.EXPECTED_PER_AREA['Baseline']
F.EXPECTED_PER_AREA['Baseline'] = 999.0
try:
    F.load_trajectory()
    raise SystemExit('THE GATE DID NOT FIRE -- do not trust the figures below')
except F.MissingQuantity as exc:
    print('gate fired, as it must:\n')
    print(exc)
finally:
    F.EXPECTED_PER_AREA['Baseline'] = real

F.load_trajectory()   # restored, and passing again
print('\nexpectation restored; loader passes.')


## 3 · Build the figure set

One command. Each builder asserts the quantities the paper states before it draws,
and raises `figstyle.MissingQuantity` naming the figure and the quantity rather
than substituting a value from a different run. A failure is reported into
`FIGURES.md` and the exit code, not swallowed.


In [ ]:
r = subprocess.run([sys.executable, 'tools/figures/make_all_figures.py'],
                   capture_output=True, text=True)
print(r.stdout)
if r.stderr:
    print('--- stderr ---')
    print(r.stderr)
print('exit code:', r.returncode)


In [ ]:
FIGDIR = REPO / 'results' / 'paper' / 'figures'
for p in sorted(FIGDIR.iterdir()):
    print(f'{p.stat().st_size:>10,} B  {p.name}')


## 4 · The figures

PNG previews, in order. The PDFs alongside them are the vector versions for the
paper — same content, embedded TrueType, no raster.


In [ ]:
from IPython.display import display, Image, Markdown

ORDER = ['F1_baseline_iso_vs_energyplus',
         'F2_baseline_energy_balance_sankey',
         'F3_correction_trajectory',
         'F4_per_correction_waterfall',
         'F5_corrected_energy_balance_sankey',
         'F6_wind_field_and_c2',
         'F7_weather_record_integrity',
         'F8_latent_gate',
         'F9_closure_residual_and_inventory',
         'F10_q50_sensitivity']

for stem in ORDER:
    png, pdf = FIGDIR / f'{stem}.png', FIGDIR / f'{stem}.pdf'
    display(Markdown(f'### `{png.name}`  ·  vector: `{pdf.name}`'))
    display(Image(filename=str(png), width=1000))


### F2 and F5 side by side

The pairing is the argument, so it is worth seeing at one scale. Both come out of
the same renderer with the same band order, the same colour map and the same
kWh-per-unit-height, so the difference between them is the model and not the
drawing: before, the five party surfaces are missing from the inventory and
96.7 kWh has no source; after, all seven surfaces are present and the balance
closes to machine zero.


In [ ]:
import base64
from IPython.display import HTML

def b64(p):
    return base64.b64encode(Path(p).read_bytes()).decode()

display(HTML(
    '<div style="display:flex;gap:8px;align-items:flex-start">'
    f'<img style="width:50%" src="data:image/png;base64,{b64(FIGDIR / "F2_baseline_energy_balance_sankey.png")}">'
    f'<img style="width:50%" src="data:image/png;base64,{b64(FIGDIR / "F5_corrected_energy_balance_sankey.png")}">'
    '</div>'))


## 5 · `FIGURES.md`

Per figure: the output files, the source files it was built from, the key numbers
it displays, and the recommended placement — so each figure can be checked against
the paper without opening the plotting code.


In [ ]:
display(Markdown((FIGDIR / 'FIGURES.md').read_text()))


## 6 · Download

Zips the whole directory — twenty files plus the index — and offers it as a single
download. On a non-Colab runtime the `files.download` call is skipped and the zip
is left on disk.


In [ ]:
import shutil

archive = shutil.make_archive('/content/AIB_paper_figures', 'zip', root_dir=str(FIGDIR))
print(archive, f'({Path(archive).stat().st_size:,} B)')

try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print('not running in Colab -- zip left on disk')


## 7 · Rebuilding one figure

Each module is runnable on its own and writes only its own outputs, so a tweak to
one figure does not cost a full rebuild.

```python
subprocess.run([sys.executable, 'tools/figures/f3_trajectory.py'])
```

Or in-process, which is faster and lets you keep the figure object:

```python
import importlib, f3_trajectory
importlib.reload(f3_trajectory)
F.apply_style()
meta = f3_trajectory.build()
display(Image(filename=str(FIGDIR / 'F3_correction_trajectory.png'), width=1000))
```

## A note on F10

F10 plots two measured points, not three. The CSIRO new-dwelling mean
$q_{50}$ = 6.9 is **not present in any committed result file** — it exists only in
the message of the recalibration commit `421c282`, on an engine tree whose own
$q_{50}$ = 4.0 row reads H = 112.70 kWh against the trajectory's 114.87 kWh. The
two sets are not commensurable, so the point is drawn as an explicit
*not measured* gap and is never interpolated. `FIGURES.md` records why.
